# Scikit-learn Multiclass Evaluation

Comprehensive evaluation of multiclass predictions against ground truth annotations.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    cohen_kappa_score,
    matthews_corrcoef,
    balanced_accuracy_score,
    top_k_accuracy_score,
    log_loss
)
from sklearn.preprocessing import LabelEncoder
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

## Load Your Data

Replace this with your actual data loading:

In [ ]:
# Option 1: Load from CSV
# df = pd.read_csv('your_data.csv')

# Option 2: Use existing DataFrame
# df = your_existing_dataframe

# Option 3: Create sample data for testing
np.random.seed(42)
n_samples = 1000
classes = ['class_A', 'class_B', 'class_C', 'class_D', 'class_E']

# Generate ground truth
annotator = np.random.choice(classes, n_samples, p=[0.3, 0.25, 0.2, 0.15, 0.1])

# Generate predictions with some errors
pred = annotator.copy()
error_mask = np.random.random(n_samples) < 0.2  # 20% error rate
pred[error_mask] = np.random.choice(classes, error_mask.sum())

# Create DataFrame
df = pd.DataFrame({
    'pred': pred,
    'annotator': annotator
})

print(f"DataFrame shape: {df.shape}")
print(f"Unique classes in ground truth: {df['annotator'].nunique()}")
print(f"Unique classes in predictions: {df['pred'].nunique()}")
df.head(10)

## Quick Data Exploration

In [ ]:
# Class distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Ground truth distribution
df['annotator'].value_counts().plot(kind='bar', ax=axes[0], color='skyblue')
axes[0].set_title('Ground Truth Distribution', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Class')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

# Prediction distribution
df['pred'].value_counts().plot(kind='bar', ax=axes[1], color='lightcoral')
axes[1].set_title('Prediction Distribution', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Class')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## Calculate All Metrics

In [ ]:
# Get predictions and ground truth
y_true = df['annotator'].to_numpy()
y_pred = df['pred'].to_numpy()

# Calculate all metrics
metrics = {}

# Basic accuracy
metrics['accuracy'] = accuracy_score(y_true, y_pred)
metrics['balanced_accuracy'] = balanced_accuracy_score(y_true, y_pred)

# Agreement metrics
metrics['cohen_kappa'] = cohen_kappa_score(y_true, y_pred)
metrics['matthews_corrcoef'] = matthews_corrcoef(y_true, y_pred)

# Per-class and averaged metrics
precision, recall, f1, support = precision_recall_fscore_support(
    y_true, y_pred, average=None, zero_division=0
)

# Macro averages
metrics['precision_macro'] = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)[0]
metrics['recall_macro'] = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)[1]
metrics['f1_macro'] = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)[2]

# Weighted averages
metrics['precision_weighted'] = precision_recall_fscore_support(y_true, y_pred, average='weighted', zero_division=0)[0]
metrics['recall_weighted'] = precision_recall_fscore_support(y_true, y_pred, average='weighted', zero_division=0)[1]
metrics['f1_weighted'] = precision_recall_fscore_support(y_true, y_pred, average='weighted', zero_division=0)[2]

# Display results
print("="*60)
print("OVERALL METRICS")
print("="*60)
print(f"Accuracy:                {metrics['accuracy']:.4f}")
print(f"Balanced Accuracy:       {metrics['balanced_accuracy']:.4f}")
print(f"Cohen's Kappa:           {metrics['cohen_kappa']:.4f}")
print(f"Matthews Corr Coef:      {metrics['matthews_corrcoef']:.4f}")

print("\n" + "="*60)
print("AVERAGED METRICS")
print("="*60)
print("\nMacro Averages (unweighted):")
print(f"  Precision: {metrics['precision_macro']:.4f}")
print(f"  Recall:    {metrics['recall_macro']:.4f}")
print(f"  F1-Score:  {metrics['f1_macro']:.4f}")

print("\nWeighted Averages (by support):")
print(f"  Precision: {metrics['precision_weighted']:.4f}")
print(f"  Recall:    {metrics['recall_weighted']:.4f}")
print(f"  F1-Score:  {metrics['f1_weighted']:.4f}")

## Classification Report

In [ ]:
# Generate classification report
print(classification_report(y_true, y_pred, zero_division=0))

## Confusion Matrix

In [ ]:
# Create confusion matrix
cm = confusion_matrix(y_true, y_pred)

# Plot confusion matrix
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=np.unique(y_true), 
            yticklabels=np.unique(y_true),
            cbar_kws={'label': 'Count'})
plt.title('Confusion Matrix', fontsize=14, fontweight='bold')
plt.xlabel('Predicted', fontsize=12)
plt.ylabel('Actual', fontsize=12)
plt.tight_layout()
plt.show()

# Calculate and display accuracy for each class
print("\nPer-Class Accuracy:")
print("="*40)
class_accuracy = cm.diagonal() / cm.sum(axis=1)
for i, cls in enumerate(np.unique(y_true)):
    print(f"{cls:<20}: {class_accuracy[i]:.4f}")

## Per-Class Performance Analysis

In [ ]:
# Create per-class metrics DataFrame
unique_classes = np.unique(y_true)
class_metrics = []

for i, cls in enumerate(unique_classes):
    cls_mask = y_true == cls
    class_metrics.append({
        'class': cls,
        'support': cls_mask.sum(),
        'precision': precision[i],
        'recall': recall[i],
        'f1': f1[i]
    })

class_df = pd.DataFrame(class_metrics)
class_df = class_df.sort_values('f1', ascending=False)

# Display table
print("Per-Class Metrics (sorted by F1-Score):")
print("="*60)
print(class_df.to_string(index=False))

# Plot per-class metrics
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Precision
class_df.plot(x='class', y='precision', kind='bar', ax=axes[0], color='skyblue', legend=False)
axes[0].set_title('Precision by Class', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Class')
axes[0].set_ylabel('Precision')
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(axis='y', alpha=0.3)

# Recall
class_df.plot(x='class', y='recall', kind='bar', ax=axes[1], color='lightcoral', legend=False)
axes[1].set_title('Recall by Class', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Class')
axes[1].set_ylabel('Recall')
axes[1].tick_params(axis='x', rotation=45)
axes[1].grid(axis='y', alpha=0.3)

# F1-Score
class_df.plot(x='class', y='f1', kind='bar', ax=axes[2], color='lightgreen', legend=False)
axes[2].set_title('F1-Score by Class', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Class')
axes[2].set_ylabel('F1-Score')
axes[2].tick_params(axis='x', rotation=45)
axes[2].grid(axis='y', alpha=0.3)

plt.suptitle('Per-Class Performance Metrics', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Error Analysis

In [ ]:
# Find misclassifications
errors = df[df['pred'] != df['annotator']].copy()

print(f"Total errors: {len(errors):,} ({len(errors)/len(df)*100:.2f}%)")
print(f"Total correct: {len(df) - len(errors):,} ({(len(df) - len(errors))/len(df)*100:.2f}%)")

if len(errors) > 0:
    # Create error pattern
    errors['error_pattern'] = errors['annotator'].astype(str) + ' → ' + errors['pred'].astype(str)
    
    # Top error patterns
    error_counts = errors['error_pattern'].value_counts().head(10)
    
    print("\nTop 10 Most Common Misclassifications:")
    print("="*50)
    for pattern, count in error_counts.items():
        percentage = count / len(errors) * 100
        print(f"{pattern:<30} : {count:>5} ({percentage:>5.1f}%)")
    
    # Plot error patterns
    plt.figure(figsize=(12, 6))
    error_counts.plot(kind='barh', color='salmon')
    plt.title('Top 10 Error Patterns', fontsize=12, fontweight='bold')
    plt.xlabel('Count')
    plt.ylabel('Error Pattern (Actual → Predicted)')
    plt.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.show()

## Export Results

In [ ]:
# Save metrics to CSV
metrics_df = pd.DataFrame([metrics])
metrics_df.to_csv('evaluation_metrics.csv', index=False)
print("Metrics saved to: evaluation_metrics.csv")

# Save per-class metrics
class_df.to_csv('per_class_metrics.csv', index=False)
print("Per-class metrics saved to: per_class_metrics.csv")

# Save confusion matrix
cm_df = pd.DataFrame(cm, index=unique_classes, columns=unique_classes)
cm_df.to_csv('confusion_matrix.csv')
print("Confusion matrix saved to: confusion_matrix.csv")

print("\n✅ All results exported successfully!")